# Proyecto Curso Intermedio de Analítica Avanzada

En este archivo se encuentran las pruebas realizadas para validar los modelos con sus parámetros optimizados. Este repositorio ha sido elaborado para poder analizar ciertas problemáticas de la NBA con modelos de machine learning y deep learning. Los autores de este proyecto son:

- Luis Alberto Arias Llaguno
- David Ceballos Mata
- Ruben

En este archivo se verá problema por problema los resultados obtenidos para validar los modelos con sus parámetros optimizados. Asimismo, se verá la exploración de datos y la limpieza de datos de la api de la NBA para poder obtener los datos necesarios para los modelos.

# 0. Importación de librerías

In [20]:
# Instalación de librerías

%pip install nba_api
%pip install tensorflow
%pip install keras
%pip install keras_tuner

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [64]:
# Importación de librerías

from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
from nba_api.stats.static import players
from nba_api.stats.endpoints import leaguedashplayerstats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, silhouette_score, r2_score
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt

import mlflow
import mlflow.tensorflow

# 1. Exploración de datos

# 2. Problemas a Resolver

## 2.1 Predicción de rendimiento partido a partido usando LSTM (Parte 1) (Luis)

### Problema: 
Predecir la línea de puntos, de un equipo (en este caso Lakers) en su próximo partido tomando en cuenta los juegos anteriores. Para este problema, vamos a probar con los 3, 5, y 10 juegos anteriores para poder predecir los puntos de los futuros juegos de los lakers. En esta parte 1 se puede ver que solo se toman en cuenta los juegos de los Lakers como características para el modelo de redes neuronales.

### 2.1.1 Obtención y de datos

In [22]:
# Obtener el ID de los Lakers
nba_teams = teams.get_teams()
lakers = [team for team in nba_teams if team['full_name'] == 'Los Angeles Lakers'][0]
lakers_id = lakers['id']

# Usar LeagueGameFinder para obtener todos los juegos de los Lakers
gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=lakers_id)
games = gamefinder.get_data_frames()[0]

# Crear DataFrame con las columnas más relevantes
df_lakers = games[[
    'SEASON_ID',           # Temporada
    'GAME_ID',             # ID del juego
    'GAME_DATE',           # Fecha del juego
    'MATCHUP',             # Matchup (LAL vs. OPP o LAL @ OPP)
    'WL',                  # Win/Loss
    'MIN',                 # Minutos jugados
    'PTS',                 # Puntos anotados
    'FGM',                 # Field Goals Made (tiros de campo anotados)
    'FGA',                 # Field Goals Attempted (tiros de campo intentados)
    'FG_PCT',              # Porcentaje de tiros de campo
    'FG3M',                # 3-Point Field Goals Made (triples anotados)
    'FG3A',                # 3-Point Field Goals Attempted (triples intentados)
    'FG3_PCT',             # Porcentaje de triples
    'FTM',                 # Free Throws Made (tiros libres anotados)
    'FTA',                 # Free Throws Attempted (tiros libres intentados)
    'FT_PCT',              # Porcentaje de tiros libres
    'OREB',                # Rebotes ofensivos
    'DREB',                # Rebotes defensivos
    'REB',                 # Rebotes totales
    'AST',                 # Asistencias
    'STL',                 # Robos
    'BLK',                 # Bloqueos
    'TOV',                 # Pérdidas de balón
    'PF',                  # Faltas personales
    'PLUS_MINUS'           # Plus/Minus
]].copy()

df_lakers.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4101 entries, 0 to 4100
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SEASON_ID   4101 non-null   object 
 1   GAME_ID     4101 non-null   object 
 2   GAME_DATE   4101 non-null   object 
 3   MATCHUP     4101 non-null   object 
 4   WL          4098 non-null   object 
 5   MIN         4101 non-null   int64  
 6   PTS         4101 non-null   int64  
 7   FGM         4101 non-null   int64  
 8   FGA         4101 non-null   int64  
 9   FG_PCT      4101 non-null   float64
 10  FG3M        4101 non-null   int64  
 11  FG3A        4101 non-null   int64  
 12  FG3_PCT     4089 non-null   float64
 13  FTM         4101 non-null   int64  
 14  FTA         4101 non-null   int64  
 15  FT_PCT      4099 non-null   float64
 16  OREB        4101 non-null   int64  
 17  DREB        4101 non-null   int64  
 18  REB         4101 non-null   int64  
 19  AST         4101 non-null  

### 2.1.2 Limpieza de datos

In [23]:
# Convertir la fecha a formato datetime
df_lakers['GAME_DATE'] = pd.to_datetime(df_lakers['GAME_DATE']) 

# Ordenar por fecha (más reciente primero)
df_lakers = df_lakers.sort_values('GAME_DATE', ascending=True).reset_index(drop=True) 

# Agregar columna para identificar si es juego en casa o de visitante
df_lakers['ES_CASA'] = df_lakers['MATCHUP'].str.contains('vs.').astype(int)

# Extraer el oponente
df_lakers['OPONENTE'] = df_lakers['MATCHUP'].str.split().str[-1]

# Eliminar plus_minus
df_lakers = df_lakers.drop(columns=['PLUS_MINUS']) 

# Eliminar filas con ANY valor nulo en df_lakers
df_lakers = df_lakers.dropna().reset_index(drop=True) 

# Crear columna WIN (1 para victoria, 0 para derrota)
df_lakers['WIN'] = (df_lakers['WL'] == 'W').astype(int)

# Contar los días que hubo entre juegos
df_lakers['DAYS_REST'] = df_lakers['GAME_DATE'].diff().dt.days
df_lakers['DAYS_REST'] = df_lakers['DAYS_REST'].fillna(df_lakers['DAYS_REST'].median()) # Rellenar valores nulos con la mediana
df_lakers['DAYS_REST'] = df_lakers['DAYS_REST'].astype(int)

# Visualizar los datos
pd.set_option('display.max_columns', None)
df_lakers.info()
df_lakers.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4086 entries, 0 to 4085
Data columns (total 28 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   SEASON_ID  4086 non-null   object        
 1   GAME_ID    4086 non-null   object        
 2   GAME_DATE  4086 non-null   datetime64[ns]
 3   MATCHUP    4086 non-null   object        
 4   WL         4086 non-null   object        
 5   MIN        4086 non-null   int64         
 6   PTS        4086 non-null   int64         
 7   FGM        4086 non-null   int64         
 8   FGA        4086 non-null   int64         
 9   FG_PCT     4086 non-null   float64       
 10  FG3M       4086 non-null   int64         
 11  FG3A       4086 non-null   int64         
 12  FG3_PCT    4086 non-null   float64       
 13  FTM        4086 non-null   int64         
 14  FTA        4086 non-null   int64         
 15  FT_PCT     4086 non-null   float64       
 16  OREB       4086 non-null   int64         


,SEASON_ID,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,ES_CASA,OPONENTE,WIN,DAYS_REST
0,21983,0028300010,1983-10-29,LAL @ UTH,W,240,120,46,98,0.469,0,1,0.000,28,39,0.718,29,31,60,25,10,10,20,41,0,UTH,1,2
1,21983,0028300035,1983-11-02,LAL @ SDC,L,240,106,43,96,0.448,1,2,0.500,19,25,0.760,16,33,49,29,14,7,24,24,0,SDC,0,4
2,21983,0028300053,1983-11-05,LAL @ DAL,L,240,102,47,97,0.485,0,2,0.000,8,12,0.667,18,23,41,29,7,8,20,27,0,DAL,0,3
3,21983,0028300066,1983-11-08,LAL @ DEN,W,240,133,49,106,0.462,2,3,0.667,33,36,0.917,22,31,53,30,11,8,23,31,0,DEN,1,3
4,21983,0028300068,1983-11-09,LAL vs. DAL,W,240,120,50,94,0.532,2,3,0.667,18,28,0.643,16,33,49,33,7,6,17,31,1,DAL,1,1


### 2.1.3 Definición de columnas de features y target y Creación de Datasets de Entrenamiento y Validación

In [66]:
feature_cols = [
    'ES_CASA',
    'WIN',
    'MIN',
    'FGM', 'FGA', 'FG_PCT',
    'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB',
    'AST', 'STL', 'BLK',
    'TOV', 'PF',
    'DAYS_REST'
]

target_col = 'PTS'

data_features = df_lakers[feature_cols].values.astype('float32')
data_target = df_lakers[target_col].values.astype('float32')

print("Features shape (sin secuencias):", data_features.shape)
print("Target shape (sin secuencias):", data_target.shape)

Features shape (sin secuencias): (4086, 21)
Target shape (sin secuencias): (4086,)


In [67]:
# Función para construir secuencias de longitud N
def build_sequences(features, target, seq_len=5):
    X, y = [], []
    n_samples = len(features)
    
    for i in range(n_samples - seq_len):
        seq_x = features[i:i+seq_len]
        seq_y = target[i+seq_len]    # Puntos del partido siguiente
        X.append(seq_x)
        y.append(seq_y)
    
    return np.array(X), np.array(y)

# Función para preparar datos para secuencias de longitud N
def prepare_data_for_seq_len(seq_len):
    X, y = build_sequences(data_features, data_target, seq_len=seq_len)
    print(f"\nSEQ_LEN = {seq_len} -> X: {X.shape}, y: {y.shape}")
    
    # split temporal 70/15/15
    n = X.shape[0]
    train_size = int(n * 0.7)
    val_size   = int(n * 0.15)
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_val   = X[train_size:train_size+val_size]
    y_val   = y[train_size:train_size+val_size]
    X_test  = X[train_size+val_size:]
    y_test  = y[train_size+val_size:]
    
    num_features = X.shape[2]
    
    # Escalar
    X_train_2d = X_train.reshape(-1, num_features)
    X_val_2d   = X_val.reshape(-1, num_features)
    X_test_2d  = X_test.reshape(-1, num_features)
    
    scaler = StandardScaler()
    X_train_scaled_2d = scaler.fit_transform(X_train_2d)
    X_val_scaled_2d   = scaler.transform(X_val_2d)
    X_test_scaled_2d  = scaler.transform(X_test_2d)
    
    X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
    X_val_scaled   = X_val_scaled_2d.reshape(X_val.shape)
    X_test_scaled  = X_test_scaled_2d.reshape(X_test.shape)
    
    return (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    )

### 2.1.4 Modelo Tuneado con Keras

In [53]:
import tensorflow as tf
from tensorflow.keras import layers, models
import keras_tuner as kt

def build_lstm_model(hp, seq_len, num_features):
    model = tf.keras.Sequential()
    
    units = hp.Int('lstm_units', min_value=16, max_value=128, step=16)
    model.add(
        layers.LSTM(
            units,
            return_sequences=False,
            input_shape=(seq_len, num_features)
        )
    )
    
    dropout_rate = hp.Float('dropout_rate', 0.0, 0.5, step=0.1)
    model.add(layers.Dropout(dropout_rate))
    
    dense_units = hp.Int('dense_units', min_value=8, max_value=64, step=8)
    model.add(layers.Dense(dense_units, activation='relu'))
    
    model.add(layers.Dense(1, activation='linear'))
    
    optimizer_name = hp.Choice('optimizer', ['sgd', 'rmsprop', 'adam'])
    learning_rate = hp.Float(
        'learning_rate',
        min_value=1e-4,
        max_value=1e-2,
        sampling='log'
    )
    
    clipnorm = hp.Choice('clipnorm', values=[0.0, 1.0, 2.0])
    clip_value = None if clipnorm == 0.0 else clipnorm
    
    if optimizer_name == 'sgd':
        momentum = hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.3)
        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=momentum,
            clipnorm=clip_value
        )
    elif optimizer_name == 'rmsprop':
        optimizer = tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate,
            clipnorm=clip_value
        )
    else:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=clip_value
        )
    
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    return model


class BatchSizeTuner(kt.Hyperband):
    """Tuner que también tunea batch_size vía hp."""
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        batch_size = hp.Choice('batch_size', [16, 32, 64])
        kwargs['batch_size'] = batch_size
        return super().run_trial(trial, *args, **kwargs)


### 2.1.5 MLflow con la búsqueda de hiperparámetros óptimos

In [56]:
import os
import shutil
from pathlib import Path
import mlflow
import mlflow.tensorflow

# 1. Carpeta raíz PARA MLflow (fuera del repo)
ml_root = Path(r"C:\Downloads\mlflow_nba")

# 2. Borrar lo que hubiera antes (si quieres empezar limpio)
shutil.rmtree(ml_root, ignore_errors=True)

# 3. Crear la carpeta raíz y la subcarpeta .trash (MUY IMPORTANTE)
ml_root.mkdir(parents=True, exist_ok=True)
trash_dir = ml_root / ".trash"
trash_dir.mkdir(exist_ok=True)

print("MLflow root:", ml_root)
print("Existe root:", ml_root.exists(), " / es dir:", ml_root.is_dir())
print("Existe .trash:", trash_dir.exists(), " / es dir:", trash_dir.is_dir())

# 4. Configurar tracking_uri para que apunte ahí
tracking_uri = ml_root.as_uri()  # file:///C:/Downloads/mlflow_nba
mlflow.set_tracking_uri(tracking_uri)
print("Tracking URI actual:", mlflow.get_tracking_uri())

# 5. Crear/usar experimento y activar autolog
mlflow.set_experiment("nba_lakers_lstm")
mlflow.tensorflow.autolog(disable=True)


2025/12/10 22:41:39 INFO mlflow.tracking.fluent: Experiment with name 'nba_lakers_lstm' does not exist. Creating a new experiment.


MLflow root: C:\Downloads\mlflow_nba
Existe root: True  / es dir: True
Existe .trash: True  / es dir: True
Tracking URI actual: file:///C:/Downloads/mlflow_nba


In [48]:
with mlflow.start_run(run_name="prueba_en_downloads"):
    mlflow.log_param("test_param", 123)
    mlflow.log_metric("test_metric", 0.99)

print("Contenido de C:\\Downloads\\mlflow_nba:", os.listdir(ml_root))
for name in os.listdir(ml_root):
    print("  ", name)


Contenido de C:\Downloads\mlflow_nba: ['.trash', '918181637898583977']
   .trash
   918181637898583977


In [49]:
from pathlib import Path
import shutil

kt_dir = Path(r"E:\001 - Portafolio Anáhuac\01 - Ingeniería en Tecnologías de la Información\4to Semestre\Analítica Avanzada\proyecto-analitica-nba") / "kt_lakers_runs"
shutil.rmtree(kt_dir, ignore_errors=True)
kt_dir.mkdir(parents=True, exist_ok=True)
base_dir = kt_dir.as_posix()

print("Keras Tuner dir:", base_dir)


Keras Tuner dir: E:/001 - Portafolio Anáhuac/01 - Ingeniería en Tecnologías de la Información/4to Semestre/Analítica Avanzada/proyecto-analitica-nba/kt_lakers_runs


In [51]:
import os
import shutil
from pathlib import Path

# Carpeta para Keras Tuner (FUERA del repo)
kt_root = Path(r"C:\Downloads\kt_lakers_lstm")

# Borramos cualquier cosa vieja
shutil.rmtree(kt_root, ignore_errors=True)

# Creamos carpeta limpia
kt_root.mkdir(parents=True, exist_ok=True)

# Usamos ruta tipo POSIX (con /) que TF maneja bien
base_dir = kt_root.as_posix()

print("Keras Tuner base_dir:", base_dir)
print("Existe:", os.path.exists(base_dir), " / es dir:", os.path.isdir(base_dir))
print("Contenido:", os.listdir(base_dir))


Keras Tuner base_dir: C:/Downloads/kt_lakers_lstm
Existe: True  / es dir: True
Contenido: []


In [ ]:
seq_len_candidates = [3, 5, 10]   # SEQ_LEN (los juegos anteriores que se requieren como contexto para predecir el siguiente)
max_epochs = 40

results = []

for seq_len in seq_len_candidates:
    (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    ) = prepare_data_for_seq_len(seq_len)

    with mlflow.start_run(run_name=f"tuning_seq{seq_len}"):

        mlflow.log_param("seq_len", seq_len)

        def hypermodel(hp):
            return build_lstm_model(hp, seq_len, num_features)

        tuner = BatchSizeTuner(
            hypermodel,
            objective='val_loss',
            max_epochs=max_epochs,
            factor=3,
            directory=base_dir,
            project_name=f"seq_{seq_len}",
            overwrite=True
        )

        stop_early = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        tuner.search(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            callbacks=[stop_early],
            verbose=1
        )

        # Mejor conjunto de hiperparámetros
        best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_val_loss = best_trial.score

        # Construir modelo ganador
        best_model = tuner.hypermodel.build(best_hp)

        history = best_model.fit(
            np.concatenate([X_train_scaled, X_val_scaled], axis=0),
            np.concatenate([y_train, y_val], axis=0),
            epochs=max_epochs,
            batch_size=best_hp.get('batch_size'),
            callbacks=[stop_early],
            verbose=0
        )

        test_loss, test_mae = best_model.evaluate(X_test_scaled, y_test, verbose=0)

        # Logear solo hiperparámetros ganadores y métricas finales
        mlflow.log_metric("best_val_loss", float(best_val_loss))
        mlflow.log_metric("test_mse", float(test_loss))
        mlflow.log_metric("test_mae", float(test_mae))

        for name, value in best_hp.values.items():
            # para evitar conflictos con nombres reservados, prefijamos
            mlflow.log_param(f"best_{name}", value)

        # Guardar en lista para imprimir resumen al final
        run_result = {
            "seq_len": seq_len,
            "best_val_loss": float(best_val_loss),
            "test_mse": float(test_loss),
            "test_mae": float(test_mae),
        }
        for name, value in best_hp.values.items():
            run_result[name] = value

        results.append(run_result)

Trial 87 Complete [00h 00m 18s]
val_loss: 161.28219604492188

Best val_loss So Far: 156.2747344970703
Total elapsed time: 00h 11m 09s


C:\Users\luis_\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


### 2.1.6 Métricas e Interpretación

In [62]:
def print_results_for_seq_len(results, seq_len):
    print("\n" + "="*80)
    print(f"======================= RESULTADOS COMPLETOS: SEQ_LEN = {seq_len} =======================")
    print("="*80)

    filtered = [r for r in results if r["seq_len"] == seq_len]
    if not filtered:
        print(f"No hay resultados para SEQ_LEN={seq_len}.")
        return

    r = filtered[0]

    # Métricas
    print(f"\nbest_val_loss : {r['best_val_loss']}")
    print(f"test_mse       : {r['test_mse']}")
    print(f"test_mae       : {r['test_mae']}")

    print("\nHiperparámetros ganadores:")
    for k, v in r.items():
        if k not in ["seq_len", "best_val_loss", "test_mse", "test_mae"]:
            print(f"  - {k}: {v}")

    print("="*80 + "\n")


# Imprimir resultados completos para cada SEQ_LEN
print_results_for_seq_len(results, 3)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 3 =======================

best_val_loss : 166.90191650390625
test_mse       : 289.3242492675781
test_mae       : 13.65479564666748

Hiperparámetros ganadores:
  - lstm_units: 128
  - dropout_rate: 0.0
  - dense_units: 64
  - optimizer: sgd
  - learning_rate: 0.00011670367524494386
  - clipnorm: 0.0
  - momentum: 0.6
  - batch_size: 16
  - tuner/epochs: 14
  - tuner/initial_epoch: 5
  - tuner/bracket: 3
  - tuner/round: 2
  - tuner/trial_id: 0035



In [ ]:
print_results_for_seq_len(results, 5)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 5 =======================

best_val_loss : 162.5537872314453
test_mse       : 206.3182830810547
test_mae       : 11.51484203338623

Hiperparámetros ganadores:
  - lstm_units: 128
  - dropout_rate: 0.0
  - dense_units: 32
  - optimizer: sgd
  - learning_rate: 0.00013434479479732051
  - clipnorm: 0.0
  - momentum: 0.6
  - batch_size: 64
  - tuner/epochs: 40
  - tuner/initial_epoch: 14
  - tuner/bracket: 3
  - tuner/round: 3
  - tuner/trial_id: 0046



In [61]:
print_results_for_seq_len(results, 10)


======================= RESULTADOS COMPLETOS: SEQ_LEN = 10 =======================

best_val_loss : 156.2747344970703
test_mse       : 199.71145629882812
test_mae       : 11.33343505859375

Hiperparámetros ganadores:
  - lstm_units: 64
  - dropout_rate: 0.0
  - dense_units: 40
  - optimizer: adam
  - learning_rate: 0.0014951135160520678
  - clipnorm: 0.0
  - momentum: 0.8999999999999999
  - batch_size: 16
  - tuner/epochs: 40
  - tuner/initial_epoch: 14
  - tuner/bracket: 2
  - tuner/round: 2
  - tuner/trial_id: 0065



En este caso podemos ver que el modelo que usa los 10 juegos anteriores para predecir el siguiente juego es el que tiene mejores métricas. Esto debido a la métrica de MAE (Mean Absolute Error), la cual con su valor de 11.3334 nos dice que el modelo promedio de error es de 11.3334 puntos. En otras palabras, esto dice que el modelo está prediciendo con un error promedio de 11.3334 puntos, osea que de su predicción está en promedio 11.3334 puntos de distancia de la realidad.

In [65]:
# R^2 score

y_pred = best_model.predict(X_test_scaled).flatten()   # predicciones
r2 = r2_score(y_test, y_pred)

print("Coeficiente de determinación R²:", r2)

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
Coeficiente de determinación R²: -0.017884254455566406


In [68]:
df_lakers['PTS'].mean()

105.26284875183553

Aquí hay que hacer un detenimiento especial en la evaluación de la métrica de R², porque al ser negativo teóricamente significa que el modelo mejora que si es que simplemente sacara el promedio de los valores de la variable objetivo para los juegos anteriores y predijeramos con eso. Esto se debe a un par de factores, principalmente que el promedio de puntos de todos los juegos (que hay en nba_api) de los Lakers es de 105.26, por lo que los juegos de los lakers son bastante estables. Otro factor que tomar en cuenta es que no hay características de los contrincantes, por lo que el modelo no tiene información sobre quién es el contrincante en el juego actual y esto también llega a afectar la precisión de la predicción.

## 2.2 Predicción de rendimiento partido a partido usando LSTM (Parte 2) (Luis)

### Problema: 
Predecir la línea de puntos, de un equipo (en este caso Lakers) en su próximo partido tomando en cuenta los juegos anteriores. Para este problema, vamos a probar con los 3, 5, y 10 juegos anteriores para poder predecir los puntos de los futuros juegos de los lakers. En esta parte 2 se puede ver que no solo se toman en cuenta las características de los Lakers en ese juego, sino que también se toman en cuenta las características de los contrincantes en el mismo juego como características para el modelo de redes neuronales. En teoría, estas nuevas características beneficiarían al modelo debido a que tener las características del contrincante en ese partido también da más contexto sobre el juego y de los contrincantes como han sido en el pasado para poder predecir con mayor exactitud el rendimiento de los Lakers en el próximo partido.

### 2.2.1 Crear dataframe con stats de los contrincantes y Lakers

In [76]:
teams_df = pd.DataFrame(nba_teams)
abbr_to_id = dict(zip(teams_df["abbreviation"], teams_df["id"]))

opp_abbrevs = df_lakers["OPONENTE"].unique()
print("Oponentes únicos:", opp_abbrevs)

# ¿Qué abreviaturas NO están en los equipos oficiales?
missing = [abbr for abbr in opp_abbrevs if abbr not in abbr_to_id]
print("Abreviaturas que no están en nba_teams:", missing)


Oponentes únicos: ['UTA' 'LAC' 'DAL' 'DEN' 'PHX' 'MIL' 'CLE' 'POR' 'OKC' 'CHI' 'GSW' 'NYK'
 'SAS' 'HOU' 'SAC' 'IND' 'BOS' 'WAS' 'ATL' 'PHI' 'BKN' 'DET' 'MIA' 'CHA'
 'ORL' 'MIN' 'MEM' 'TOR' 'NOP']
Abreviaturas que no están en nba_teams: []


In [77]:
# Mapa manual de abreviaturas raras a abreviaturas oficiales de nba_api
abbr_fix_map = {
    "UTH": "UTA",  # Utah Jazz
    "SDC": "LAC",  # San Diego Clippers -> LA Clippers
    "SEA": "OKC",  # Seattle SuperSonics -> Oklahoma City Thunder
    "GOS": "GSW",  # Golden State (a veces codificado raro) -> Golden State Warriors
    "SAN": "SAS",  # San Antonio -> San Antonio Spurs
    "KCK": "SAC",  # Kansas City Kings -> Sacramento Kings
    "PHL": "PHI",  # Philadelphia (viejo código) -> Philadelphia 76ers
    "NJN": "BKN",  # New Jersey Nets -> Brooklyn Nets
    "CHH": "CHA",  # Charlotte Hornets (viejos) -> Charlotte Hornets actuales
    "VAN": "MEM",  # Vancouver Grizzlies -> Memphis Grizzlies
    "NOH": "NOP",  # New Orleans Hornets -> New Orleans Pelicans
    "NOK": "NOP",  # New Orleans/Oklahoma City Hornets -> Pelicans
}

teams_df = pd.DataFrame(nba_teams)
abbr_to_id = dict(zip(teams_df["abbreviation"], teams_df["id"]))

# Aplicar corrección directamente al dataframe de Lakers
df_lakers["OPONENTE"] = df_lakers["OPONENTE"].replace(abbr_fix_map)

# Volver a calcular oponentes únicos
opp_abbrevs = df_lakers["OPONENTE"].unique()
print("Oponentes únicos normalizados:", opp_abbrevs)

# Verificar si ya todos existen
missing = [abbr for abbr in opp_abbrevs if abbr not in abbr_to_id]
print("Abreviaturas fuera de catálogo después de normalizar:", missing)

if missing:
    print("Eliminando partidos contra equipos no-NBA:", missing)
    df_lakers = df_lakers[~df_lakers["OPONENTE"].isin(missing)].reset_index(drop=True)


Oponentes únicos normalizados: ['UTA' 'LAC' 'DAL' 'DEN' 'PHX' 'MIL' 'CLE' 'POR' 'OKC' 'CHI' 'GSW' 'NYK'
 'SAS' 'HOU' 'SAC' 'IND' 'BOS' 'WAS' 'ATL' 'PHI' 'BKN' 'DET' 'MIA' 'CHA'
 'ORL' 'MIN' 'MEM' 'TOR' 'NOP']
Abreviaturas fuera de catálogo después de normalizar: []


In [86]:
# Esta vez pedimos TODOS los juegos donde el oponente fue LAL
opp_gamefinder = leaguegamefinder.LeagueGameFinder(
    vs_team_id_nullable=lakers_id   # <-- clave: vs_team_id, no team_id
)
df_opp_raw = opp_gamefinder.get_data_frames()[0]
print("df_opp_raw (todos los rivales vs LAL) shape:", df_opp_raw.shape)

# Nos quedamos solo con los GAME_ID que están en df_lakers (por si acaso)
df_opp_raw = df_opp_raw[df_opp_raw["GAME_ID"].isin(df_lakers["GAME_ID"])]
print("df_opp_raw filtrado shape:", df_opp_raw.shape)

# Columnas relevantes del rival (mismas métricas que Lakers)
opp_cols = [
    "GAME_ID",
    "WL",          # Win/Loss del rival
    "MIN",
    "PTS",
    "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "STL", "BLK",
    "TOV", "PF",
]

df_opp = df_opp_raw[opp_cols].copy()

# Crear columna OPP_WIN (1 si el rival ganó, 0 si perdió)
df_opp["OPP_WIN"] = (df_opp["WL"] == "W").astype(int)
df_opp = df_opp.drop(columns=["WL"])

# Renombrar métricas del rival con prefijo OPP_
rename_map = {
    col: f"OPP_{col}" for col in df_opp.columns if col != "GAME_ID"
}
df_opp.rename(columns=rename_map, inplace=True)

print("df_opp shape (una fila por equipo-rival en cada juego):", df_opp.shape)
df_opp.head()


df_opp_raw (todos los rivales vs LAL) shape: (4099, 28)
df_opp_raw filtrado shape: (4082, 28)
df_opp shape (una fila por equipo-rival en cada juego): (4082, 21)


,GAME_ID,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
1,0022500362,240,108,36,102,0.353,13,36,0.361,23,26,0.885,17,37,54,21,5,4,6,17,0
2,0022500338,239,126,46,84,0.548,24,45,0.533,10,12,0.833,8,30,38,31,9,5,14,20,1
3,0022500336,240,120,44,89,0.494,15,37,0.405,17,21,0.810,14,27,41,39,9,8,15,20,0
4,0022500317,240,125,52,92,0.565,17,39,0.436,4,8,0.500,4,29,33,35,16,1,11,16,1
5,0022500308,239,121,46,91,0.505,13,26,0.500,16,20,0.800,8,32,40,23,6,1,8,27,0


In [87]:
df_opp.info()
df_opp.head()


<class 'pandas.core.frame.DataFrame'>
Index: 4082 entries, 1 to 4097
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GAME_ID      4082 non-null   object 
 1   OPP_MIN      4082 non-null   int64  
 2   OPP_PTS      4082 non-null   int64  
 3   OPP_FGM      4082 non-null   int64  
 4   OPP_FGA      4082 non-null   int64  
 5   OPP_FG_PCT   4082 non-null   float64
 6   OPP_FG3M     4082 non-null   int64  
 7   OPP_FG3A     4082 non-null   int64  
 8   OPP_FG3_PCT  4043 non-null   float64
 9   OPP_FTM      4082 non-null   int64  
 10  OPP_FTA      4082 non-null   int64  
 11  OPP_FT_PCT   4082 non-null   float64
 12  OPP_OREB     4082 non-null   int64  
 13  OPP_DREB     4082 non-null   int64  
 14  OPP_REB      4082 non-null   int64  
 15  OPP_AST      4082 non-null   int64  
 16  OPP_STL      4082 non-null   int64  
 17  OPP_BLK      4082 non-null   int64  
 18  OPP_TOV      4082 non-null   int64  
 19  OPP_PF     

,GAME_ID,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
1,0022500362,240,108,36,102,0.353,13,36,0.361,23,26,0.885,17,37,54,21,5,4,6,17,0
2,0022500338,239,126,46,84,0.548,24,45,0.533,10,12,0.833,8,30,38,31,9,5,14,20,1
3,0022500336,240,120,44,89,0.494,15,37,0.405,17,21,0.810,14,27,41,39,9,8,15,20,0
4,0022500317,240,125,52,92,0.565,17,39,0.436,4,8,0.500,4,29,33,35,16,1,11,16,1
5,0022500308,239,121,46,91,0.505,13,26,0.500,16,20,0.800,8,32,40,23,6,1,8,27,0


In [88]:
# Dataframe combinado Lakers + Rival

df_lakers_opp = df_lakers.merge(
    df_opp,
    on="GAME_ID",
    how="left",
    validate="one_to_one"  # Si truena aquí es porque hay duplicados, se puede quitar el validate
)

print("df_lakers_opp shape:", df_lakers_opp.shape)
df_lakers_opp.head()

df_lakers_opp shape: (4082, 48)


,SEASON_ID,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,ES_CASA,OPONENTE,WIN,DAYS_REST,OPP_MIN,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,OPP_PF,OPP_OPP_WIN
0,21983,0028300010,1983-10-29,LAL @ UTH,W,240,120,46,98,0.469,0,1,0.000,28,39,0.718,29,31,60,25,10,10,20,41,0,UTA,1,2,240,115,35,77,0.455,1,3,0.333,44,60,0.733,15,26,41,21,8,10,17,27,0
1,21983,0028300035,1983-11-02,LAL @ SDC,L,240,106,43,96,0.448,1,2,0.500,19,25,0.760,16,33,49,29,14,7,24,24,0,LAC,0,4,240,110,47,91,0.516,0,1,0.000,16,23,0.696,11,31,42,30,12,8,20,24,1
2,21983,0028300053,1983-11-05,LAL @ DAL,L,240,102,47,97,0.485,0,2,0.000,8,12,0.667,18,23,41,29,7,8,20,27,0,DAL,0,3,240,107,43,97,0.443,0,2,0.000,21,26,0.808,22,26,48,26,9,4,16,20,1
3,21983,0028300066,1983-11-08,LAL @ DEN,W,240,133,49,106,0.462,2,3,0.667,33,36,0.917,22,31,53,30,11,8,23,31,0,DEN,1,3,240,124,44,96,0.458,0,8,0.000,36,42,0.857,14,30,44,28,15,4,24,26,0
4,21983,0028300068,1983-11-09,LAL vs. DAL,W,240,120,50,94,0.532,2,3,0.667,18,28,0.643,16,33,49,33,7,6,17,31,1,DAL,1,1,240,106,34,83,0.410,0,2,0.000,38,46,0.826,13,30,43,22,7,2,18,23,0


### 2.2.2 Definir columnas de features y target y Creación de Datasets de Entrenamiento y Validación

In [89]:
feature_cols_22 = [
    # Features de los Lakers (igual que en 2.1)
    'ES_CASA',
    'WIN',
    'MIN',
    'FGM', 'FGA', 'FG_PCT',
    'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB',
    'AST', 'STL', 'BLK',
    'TOV', 'PF',
    'DAYS_REST',
    
    # Features del rival (prefijo OPP_)
    'OPP_MIN',
    'OPP_PTS',
    'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
    'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT',
    'OPP_FTM', 'OPP_FTA', 'OPP_FT_PCT',
    'OPP_OREB', 'OPP_DREB', 'OPP_REB',
    'OPP_AST', 'OPP_STL', 'OPP_BLK',
    'OPP_TOV', 'OPP_PF'
]

target_col_22 = 'PTS' 

data_features_22 = df_lakers_opp[feature_cols_22].values.astype('float32')
data_target_22 = df_lakers_opp[target_col_22].values.astype('float32')

print("Features 2.2 shape (sin secuencias):", data_features_22.shape)
print("Target 2.2 shape (sin secuencias):", data_target_22.shape)


Features 2.2 shape (sin secuencias): (4082, 40)
Target 2.2 shape (sin secuencias): (4082,)


### 2.2.3 Preparación de datos ecuenciales para el nuevo modelo

In [90]:
def prepare_data_for_seq_len_22(seq_len):
    X, y = build_sequences(data_features_22, data_target_22, seq_len=seq_len)
    print(f"\n[2.2] SEQ_LEN = {seq_len} -> X: {X.shape}, y: {y.shape}")
    
    n = X.shape[0]
    train_size = int(n * 0.7)
    val_size   = int(n * 0.15)
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_val   = X[train_size:train_size+val_size]
    y_val   = y[train_size:train_size+val_size]
    X_test  = X[train_size+val_size:]
    y_test  = y[train_size+val_size:]
    
    num_features = X.shape[2]
    
    # Escalado (igual que en 2.1)
    X_train_2d = X_train.reshape(-1, num_features)
    X_val_2d   = X_val.reshape(-1, num_features)
    X_test_2d  = X_test.reshape(-1, num_features)
    
    scaler = StandardScaler()
    X_train_scaled_2d = scaler.fit_transform(X_train_2d)
    X_val_scaled_2d   = scaler.transform(X_val_2d)
    X_test_scaled_2d  = scaler.transform(X_test_2d)
    
    X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
    X_val_scaled   = X_val_scaled_2d.reshape(X_val.shape)
    X_test_scaled  = X_test_scaled_2d.reshape(X_test.shape)
    
    return (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    )


### 2.2.4 Flujo de Keras Tuner y MLflow

In [ ]:
mlflow.set_experiment("nba_lakers_lstm_con_rival")  # nuevo experimento para Parte 2.2

seq_len_candidates_22 = [3, 5, 10]
max_epochs_22 = 40

results_22 = []

for seq_len in seq_len_candidates_22:
    (
        X_train_scaled, y_train,
        X_val_scaled, y_val,
        X_test_scaled, y_test,
        num_features, scaler
    ) = prepare_data_for_seq_len_22(seq_len)

    with mlflow.start_run(run_name=f"tuning_seq{seq_len}_rival"):

        mlflow.log_param("seq_len", seq_len)

        def hypermodel(hp):
            return build_lstm_model(hp, seq_len, num_features)

        tuner = BatchSizeTuner(
            hypermodel,
            objective='val_loss',
            max_epochs=max_epochs_22,
            factor=3,
            directory=base_dir,
            project_name=f"seq_{seq_len}_rival",
            overwrite=True
        )

        stop_early = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        tuner.search(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            callbacks=[stop_early],
            verbose=1
        )

        # Mejor conjunto de hiperparámetros
        best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_val_loss = best_trial.score

        # Construir modelo ganador y reentrenar con train+val
        best_model = tuner.hypermodel.build(best_hp)

        history = best_model.fit(
            np.concatenate([X_train_scaled, X_val_scaled], axis=0),
            np.concatenate([y_train, y_val], axis=0),
            epochs=max_epochs_22,
            batch_size=best_hp.get('batch_size'),
            callbacks=[stop_early],
            verbose=0
        )

        test_loss, test_mae = best_model.evaluate(X_test_scaled, y_test, verbose=0)

        # Log en MLflow
        mlflow.log_metric("best_val_loss", float(best_val_loss))
        mlflow.log_metric("test_mse", float(test_loss))
        mlflow.log_metric("test_mae", float(test_mae))

        for name, value in best_hp.values.items():
            mlflow.log_param(f"best_{name}", value)

        run_result = {
            "seq_len": seq_len,
            "best_val_loss": float(best_val_loss),
            "test_mse": float(test_loss),
            "test_mae": float(test_mae),
        }
        for name, value in best_hp.values.items():
            run_result[name] = value

        results_22.append(run_result)

### 2.2.5 Métricas y Evaluación

In [ ]:
print_results_for_seq_len(results_22, 3)

In [ ]:
print_results_for_seq_len(results_22, 5)

In [ ]:
print_results_for_seq_len(results_22, 10)

In [ ]:
y_pred_22 = best_model.predict(X_test_scaled).flatten()
r2_22 = r2_score(y_test, y_pred_22)
print("R² (modelo con stats del rival):", r2_22)